# AMP Error Analysis — Manual Inspection Workspace

This exploratory notebook loads canonical case-level error rows produced by `src/experiments/11_evaluate_amp.py`. It supports transparent filtering and disagreement inspection without recalculating benchmark metrics.

Selections here are post-hoc and must not be used to alter or rerun the frozen primary benchmark. Case examples chosen for publication require a documented, non-cherry-picked sampling rule.

## 1. Setup and canonical-artifact contract

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

EXPECTED_METHODS = ("M1", "M2", "M3", "M4")


def locate_repo_root() -> Path:
    """Locate the repository without relying on the notebook launch directory."""
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/11_evaluate_amp.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
METRICS_ROOT = REPO_ROOT / "outputs/metrics"


def load_json(path: Path) -> dict:
    if not path.is_file():
        display(Markdown(f"> **Pending:** `{path.relative_to(REPO_ROOT)}` does not exist."))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv(path: Path, required_columns=()) -> pd.DataFrame:
    """Load an evaluator table, reporting absence without synthesizing results."""
    if not path.is_file():
        display(Markdown(f"> **Pending:** `{path.relative_to(REPO_ROOT)}` does not exist."))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing canonical columns: {sorted(missing)}")
    return frame


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def show_or_pending(frame: pd.DataFrame, message="No canonical rows are available yet."):
    if frame.empty:
        display(Markdown(f"> **Pending:** {message}"))
    else:
        display(frame)


def methods_complete(manifest: dict, evaluation: str) -> bool:
    methods = manifest.get("evaluations", {}).get(evaluation, {}).get("methods", [])
    return set(methods) == set(EXPECTED_METHODS)


manifest_path = METRICS_ROOT / "amp_evaluation_manifest.json"
evaluation_manifest = load_json(manifest_path)


## 2. Artifact availability and completion gate

In [ ]:
canonical_inputs = [
    METRICS_ROOT / "amp_evaluation_manifest.json",
    METRICS_ROOT / "a1/amp_primary_results.csv",
    METRICS_ROOT / "a1/amp_per_label.csv",
    METRICS_ROOT / "a1/amp_bootstrap_cis.csv",
    METRICS_ROOT / "a1/amp_case_level_errors.csv",
    METRICS_ROOT / "a2/amp_primary_results.csv",
    METRICS_ROOT / "a2/amp_per_fold.csv",
    METRICS_ROOT / "a2/amp_per_label.csv",
    METRICS_ROOT / "a2/amp_per_jurisdiction.csv",
    METRICS_ROOT / "a2/amp_bootstrap_cis.csv",
    METRICS_ROOT / "a2/amp_case_level_errors.csv",
    METRICS_ROOT / "amp_a1_to_a2_deltas.csv",
]
availability = pd.DataFrame(
    {
        "artifact": [str(path.relative_to(REPO_ROOT)) for path in canonical_inputs],
        "available": [path.is_file() for path in canonical_inputs],
    }
)
display(availability)

gate = evaluation_manifest.get("final_completion_gate", "PENDING")
complete = (
    gate == "PASSED_M1_M2_M3_M4_A1_A2"
    and methods_complete(evaluation_manifest, "A1")
    and methods_complete(evaluation_manifest, "A2")
)
if complete:
    display(Markdown("**Canonical completion gate: PASSED for M1-M4 in A1 and A2.**"))
else:
    display(Markdown(
        "> **Incomplete benchmark:** canonical M1-M4 A1/A2 outputs are not complete. "
        "Any available rows are technical previews, not the final comparison."
    ))


## 3. Load canonical case-level evaluator outputs

In [ ]:
error_columns = (
    "method", "case_id", "search_rank", "jurisdiction", "split", "fold", "fact_summary",
    "silver_reference_amp_json", "predicted_amp_json", "false_positive_labels_json",
    "false_negative_labels_json", "exact_set_correct", "example_jaccard", "truncated_input",
)
a1_errors = load_csv(METRICS_ROOT / "a1/amp_case_level_errors.csv", error_columns).assign(evaluation="A1")
a2_errors = load_csv(METRICS_ROOT / "a2/amp_case_level_errors.csv", error_columns).assign(evaluation="A2")
case_errors = pd.concat([a1_errors, a2_errors], ignore_index=True)
case_errors["narrative_char_count"] = case_errors["fact_summary"].fillna("").str.len()
if not case_errors.empty:
    display(case_errors.groupby(["evaluation", "method"], dropna=False).size().rename("rows").reset_index())
else:
    show_or_pending(case_errors)


## 4. Reusable manual filters

In [ ]:
# Edit these controls, then rerun this cell. None of them changes a model or metric.
EVALUATION = "A1"          # "A1", "A2", or None
METHOD = None              # "M1", "M2", "M3", "M4", or None
FOLD = None                # 1, 2, 3, or None
JURISDICTION = None        # exact string or None
EXACT_SET_FAILURES_ONLY = False
TRUNCATED_M2_ONLY = False
MIN_JACCARD = None
MAX_JACCARD = None
LABEL_ID = None            # exact ontology ID found in FP or FN arrays, or None
SORT = "example_jaccard"   # example_jaccard, narrative_char_count, or search_rank
ASCENDING = True
MAX_ROWS = 100

filtered = case_errors.copy()
if EVALUATION is not None:
    filtered = filtered.loc[filtered["evaluation"].eq(EVALUATION)]
if METHOD is not None:
    filtered = filtered.loc[filtered["method"].eq(METHOD)]
if FOLD is not None:
    filtered = filtered.loc[filtered["fold"].eq(FOLD)]
if JURISDICTION is not None:
    filtered = filtered.loc[filtered["jurisdiction"].eq(JURISDICTION)]
if EXACT_SET_FAILURES_ONLY:
    filtered = filtered.loc[filtered["exact_set_correct"].eq(0)]
if TRUNCATED_M2_ONLY:
    filtered = filtered.loc[filtered["method"].eq("M2") & filtered["truncated_input"].eq(1)]
if MIN_JACCARD is not None:
    filtered = filtered.loc[filtered["example_jaccard"].ge(MIN_JACCARD)]
if MAX_JACCARD is not None:
    filtered = filtered.loc[filtered["example_jaccard"].le(MAX_JACCARD)]
if LABEL_ID is not None:
    def contains_label(value):
        return LABEL_ID in json.loads(value)
    filtered = filtered.loc[
        filtered["false_positive_labels_json"].map(contains_label)
        | filtered["false_negative_labels_json"].map(contains_label)
    ]

inspection_columns = [
    "evaluation", "method", "search_rank", "jurisdiction", "fold", "narrative_char_count",
    "exact_set_correct", "example_jaccard", "truncated_input", "silver_reference_amp_json",
    "predicted_amp_json", "false_positive_labels_json", "false_negative_labels_json", "fact_summary",
]
display(filtered.sort_values(SORT, ascending=ASCENDING)[inspection_columns].head(MAX_ROWS))


## 5. Prediction disagreements among M1–M4

In [ ]:
if not case_errors.empty:
    disagreement_key = ["evaluation", "search_rank", "jurisdiction", "fold"]
    prediction_matrix = case_errors.pivot_table(
        index=disagreement_key,
        columns="method",
        values="predicted_amp_json",
        aggfunc="first",
    ).reset_index()
    method_columns = [method for method in EXPECTED_METHODS if method in prediction_matrix.columns]
    prediction_matrix["distinct_prediction_count"] = prediction_matrix[method_columns].nunique(axis=1, dropna=True)
    disagreements = prediction_matrix.loc[prediction_matrix["distinct_prediction_count"].gt(1)]
    display(disagreements.sort_values(["evaluation", "search_rank"]).head(100))
else:
    display(Markdown("> **Pending:** canonical case-level rows are unavailable."))


## 6. M3 versus M4 and supervised versus LLM views

In [ ]:
if "prediction_matrix" in globals() and not prediction_matrix.empty:
    if {"M3", "M4"}.issubset(prediction_matrix.columns):
        m3_m4 = prediction_matrix.loc[
            prediction_matrix["M3"].notna()
            & prediction_matrix["M4"].notna()
            & prediction_matrix["M3"].ne(prediction_matrix["M4"])
        ]
        display(Markdown("**M3/M4 prediction disagreements:**"))
        display(m3_m4.head(100))
    else:
        display(Markdown("> **Pending:** both M3 and M4 canonical rows are required."))

    if set(EXPECTED_METHODS).issubset(prediction_matrix.columns):
        supervised_llm = prediction_matrix.loc[
            prediction_matrix[["M1", "M2"]].nunique(axis=1).eq(1)
            & prediction_matrix[["M3", "M4"]].nunique(axis=1).eq(1)
            & prediction_matrix["M1"].ne(prediction_matrix["M3"])
        ]
        display(Markdown("**Cases where supervised methods agree, LLM methods agree, and the groups disagree:**"))
        display(supervised_llm.head(100))
    else:
        display(Markdown("> **Pending:** all four canonical method rows are required for the grouped view."))


## 7. Canonical jurisdiction performance and M2 truncation cases

In [ ]:
a2_jurisdiction = load_csv(
    METRICS_ROOT / "a2/amp_per_jurisdiction.csv",
    ("method", "jurisdiction", "fold", "macro_f1", "micro_f1", "exact_set_accuracy", "example_jaccard", "test_n"),
)
show_or_pending(a2_jurisdiction)

if not case_errors.empty:
    m2_truncated = case_errors.loc[case_errors["method"].eq("M2") & case_errors["truncated_input"].eq(1)]
    display(Markdown("**Canonical M2 rows marked as truncated:**"))
    display(m2_truncated[inspection_columns].sort_values(["evaluation", "search_rank"]))


## 8. Rare-label inspection

In [ ]:
a1_label = load_csv(METRICS_ROOT / "a1/amp_per_label.csv", ("method", "label_id", "support", "status"))
a2_label = load_csv(METRICS_ROOT / "a2/amp_per_label.csv", ("method", "label_id", "support", "status"))
label_support = pd.concat([a1_label.assign(evaluation="A1"), a2_label.assign(evaluation="A2")], ignore_index=True)
if not label_support.empty:
    display(label_support.sort_values(["evaluation", "support", "label_id"]))
display(Markdown(
    "Set `LABEL_ID` in the manual-filter cell to inspect errors involving a rare label. "
    "In A2, zero-support Organ Removal is N/A for per-label F1 but false positives remain visible in case-level errors."
))


## 9. Researcher notes and post-hoc labeling

Record the filter settings and selection rule for every saved case set. Mark any analysis developed after viewing test outcomes as **post-hoc exploratory analysis**. Do not use this notebook to tune or rerun the frozen benchmark, and do not infer narrative-grounded correctness from disagreement with the silver reference.